In [ ]:
import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fastf1 import plotting
from src.plotset import setup_plot
from src.plotset import save_fig

setup_plot()

In [ ]:
fastf1.Cache.enable_cache('./f1_cache')
fastf1.Cache.get_cache_info()

In [ ]:
session = fastf1.get_session(2026, 3, 'R')
session.load()

In [ ]:
drivers = session.drivers
drivers = [session.get_driver(drv).Abbreviation for drv in drivers]

In [ ]:
comp_palette = {'1_SOFT': '#da291c',
                '2_SOFT': '#da291c',
                '3_SOFT': '#da291c',
                '1_MEDIUM': '#ffd12e',
                '2_MEDIUM': '#ffd12e',
                '3_MEDIUM': '#ffd12e',
                '1_HARD': '#f0f0ec',
                '2_HARD': '#f0f0ec',
                '3_HARD': '#f0f0ec'
}

In [ ]:
setup_plot(xyticksize=24,axeslabel=28, figtitle=40, legendfont=28)

fig, ax = plt.subplots(figsize=(15,8))

for drv in drivers[:6]:

    df = session.laps.pick_drivers(drv).pick_quicklaps(1.03).copy()
    df['LapTime'] = df['LapTime'].dt.total_seconds()
    df['Stint_Compound'] = df.apply(lambda x: str(int(x['Stint'])) + '_'+ x['Compound'],axis=1)

    color = plotting.get_driver_color(session=session,identifier=drv)

    sns.violinplot(df,x='Driver',y='LapTime',hue='Stint_Compound',palette=[color]*df.Stint.nunique(),
                inner=None,linecolor='w',legend=False)
    sns.stripplot(df,x='Driver',y='LapTime',hue='Stint_Compound',palette=comp_palette
                ,jitter=0.08,linewidth=1,size=15,edgecolor='k',dodge=True,legend=False)

# Collect laps
mean_laps = (
    session.laps.pick_drivers(drivers[:6])
    .pick_quicklaps(1.03)
    .copy()
)
mean_laps['LapTime'] = mean_laps['LapTime'].dt.total_seconds()

# Mean per driver+stint
driver_stint_means = mean_laps.groupby(["Driver", "Stint"])["LapTime"].mean().reset_index()

# Pivot: rows=Driver, cols=Stint
pivot_means = driver_stint_means.pivot(index="Driver", columns="Stint", values="LapTime")
pivot_means = pivot_means.loc[['ANT','PIA','LEC','RUS','NOR','HAM']]

# Plot one line per stint across drivers
for stint in pivot_means.columns[:1]:
    ax.plot(
        [-0.2,0.8,1.8,2.8,3.8,4.8],            # x = driver index
        pivot_means[stint].values,          # y = mean laptimes for that stint
        marker="o", linestyle="--", linewidth=2, markersize=10,
        label=f"Stint {int(stint)} Avg Lap Time"
    )

# Plot one line per stint across drivers
for stint in pivot_means.columns[1:]:
    ax.plot(
        [0.2,1.2,2.2,3.2,4.2,5.2],            # x = driver index
        pivot_means[stint].values,          # y = mean laptimes for that stint
        marker="o", linestyle="--", linewidth=2, markersize=10,
        label=f"Stint {int(stint)} Avg Lap Time"
    )

ax.legend(loc='lower right')
ax.set_ylim(91.5,96)
ax.set_ylabel("Lap Time (sec)")
ax.set_xlabel("")
ax.set_xticks([])
ax.set_yticks(np.arange(92,97,1))
ax.set_title('Race Pace (lower laptime is better)',pad=30)

ax.grid(visible=False)

In [ ]:
save_fig(fig, name='race_pace', loc='Reel28', trs=True)

In [ ]:
rus = session.laps.pick_drivers('RUS').copy()
rus['LapEndTime'] = rus['Time'].dt.total_seconds()
rus = rus.set_index('LapNumber')['LapEndTime']

In [ ]:
ant = session.laps.pick_drivers('ANT').copy()
ant['LapEndTime'] = ant['Time'].dt.total_seconds()
ant = ant.set_index('LapNumber')['LapEndTime']

In [ ]:
lec = session.laps.pick_drivers('LEC').copy()
lec['LapEndTime'] = lec['Time'].dt.total_seconds()
lec = lec.set_index('LapNumber')['LapEndTime']

In [ ]:
ham = session.laps.pick_drivers('HAM').copy()
ham['LapEndTime'] = ham['Time'].dt.total_seconds()
ham = ham.set_index('LapNumber')['LapEndTime']

In [ ]:
nor = session.laps.pick_drivers('NOR').copy()
nor['LapEndTime'] = nor['Time'].dt.total_seconds()
nor = nor.set_index('LapNumber')['LapEndTime']

In [ ]:
pia = session.laps.pick_drivers('PIA').copy()
pia['LapEndTime'] = pia['Time'].dt.total_seconds()
pia = pia.set_index('LapNumber')['LapEndTime']

In [ ]:
g_rus = rus - ant
g_lec = lec - ant
g_ham = ham - ant
g_nor = nor - ant
g_pia = pia - ant

In [ ]:
setup_plot(xyticksize=24, axeslabel=28, figtitle=40, legendfont=25)

fig, ax = plt.subplots(figsize = (15,8))

ax.axhline(y=0,color='w',ls='--',lw=3)

ax.plot(g_pia,color=plotting.get_driver_color(identifier='PIA',session=session), ls='-',lw=3,label='PIASTRI')
ax.plot(g_lec,color=plotting.get_driver_color(identifier='LEC',session=session), ls='-',lw=3,label='LECLERC')
ax.plot(g_rus,color=plotting.get_driver_color(identifier='RUS',session=session), ls='--',lw=3,label='RUSSELL')
ax.plot(g_nor,color=plotting.get_driver_color(identifier='NOR',session=session), ls='--',lw=3,label='NORRIS')
ax.plot(g_ham,color=plotting.get_driver_color(identifier='HAM',session=session), ls='--',lw=3,label='HAMILTON')

ax.fill_between(x=[22,27],y1=-20,y2=35,color='yellow',alpha=0.3,label='SAFETY CAR')

ax.set_xlabel('Lap Number')
ax.set_ylabel('Gap in seconds')
ax.set_title('Gap to leader (Antonelli)')

ax.set_xlim(1,51)
ax.set_ylim(-10,35)

ax.legend(loc=(0.01,0.48))
ax.grid(visible=False)

In [ ]:
save_fig(fig, name='race_trace', loc='Reel28', trs=True)

In [ ]:
sc = session.laps.pick_drivers('ANT')[['LapNumber', 'TrackStatus']]
sc = sc[sc.TrackStatus != '1']

In [ ]:
sc